# Data Science Programming - Sesion 2 (via avanzada)

**Especializacion en Ciencia de Datos - Universidad Santo Tomas - Tunja**
**Prof. Hugo Hernandez | Martes 1 de septiembre, 2026**

---

## Antes de empezar, lee esto

Este notebook **reemplaza** al de la sesion. Si ya programas, trabaja aqui a tu ritmo
mientras el resto del grupo ve NumPy desde cero.

- **No vale nota.** Existe para que no te toque esperar.
- **Trabajas solo.** Durante la clase estare con quienes estan empezando, asi que este
  notebook trae todo lo que necesitas. Cada reto se verifica solo: corres la celda de
  verificacion y sabes si lo lograste.
- **No tienes que entregarlo.** Estos retos no se califican. Subelo al aula solo si quieres que lo revise y te comente por escrito: es totalmente opcional y no hay que exponer.
- Si un reto se te atraviesa, sigue con el siguiente y vuelve despues.

Cubre el mismo temario que la clase normal - arrays, vectorizacion, agregacion, filtrado
booleano, `axis` - pero llevado hasta donde de verdad se usa. Al final hay un puente con lo
minimo de pandas y matplotlib que necesitas para la **Actividad 1**, que se entrega
despues de esta sesion.

**Una regla que atraviesa todo el notebook: no se permiten bucles de Python.** Ni `for`,
ni `while`, ni comprensiones sobre los elementos del array. Ningun assert puede
verificarlo, asi que va por tu cuenta. Pero cada reto esta disenado para que la version con
bucles sea larga, fea y lenta, y la version vectorizada quepa en tres o cuatro lineas. Si
te sale un bucle, es que te falto una funcion de NumPy.

---
## Repaso relampago

Todo lo que la clase normal ve en la primera hora, en dos tablas.

| Pieza | Que hace |
|---|---|
| `np.array([[1,2],[3,4]])` | Crea el array. Tipo homogeneo, tamano fijo |
| `a.shape` / `a.dtype` / `a.ndim` | Forma, tipo de dato, numero de ejes |
| `np.zeros(n)`, `np.ones((r,c))`, `np.full(sh,v)` | Fabricas de arrays |
| `np.arange(a,b,paso)` / `np.linspace(a,b,n)` | Rangos por paso / por cantidad |
| `a * 2`, `a + b`, `a ** 2` | Vectorizacion: opera elemento a elemento, sin bucles |
| `a.sum()`, `a.mean()`, `a.std()`, `a.min()`, `a.max()` | Agregacion sobre todo el array |
| `a.sum(axis=0)` / `a.sum(axis=1)` | El eje que **colapsa**: 0 recorre filas (resultado por columna), 1 recorre columnas (resultado por fila) |
| `a[a > 10]` | Filtrado booleano: devuelve los valores que cumplen |
| `(a > 10) & (a < 20)` | Condiciones compuestas. `&` `|` `~`, nunca `and` `or` `not`, y con parentesis |
| `a[1, 2]`, `a[0]`, `a[:, 3]` | Fila 1 columna 2, fila 0 completa, columna 3 completa |

| Pieza que la clase normal no alcanza a ver | Para que |
|---|---|
| `np.where(cond, si, no)` | Un `if/else` vectorizado. Se puede anidar |
| `keepdims=True` | Que el agregado conserve el eje: `(6,1)` en vez de `(6,)`. Clave para restar despues |
| `a[:, np.newaxis]` o `a[:, None]` | Convierte `(6,)` en `(6,1)` para forzar broadcasting por filas |
| `np.argsort`, `np.argmax`, `np.argmin` | Devuelven **indices**, no valores |
| `a[filas, columnas]` | Indexado avanzado: pide posiciones arbitrarias de una |
| `np.cumsum` | Suma acumulada. La base de las ventanas moviles sin bucles |
| `np.errstate`, `np.isnan`, `np.nan` | Control de division por cero y de faltantes |

**Broadcasting en una frase:** NumPy alinea las formas por la derecha; dos dimensiones son
compatibles si son iguales o si una de ellas vale 1. `(6,12)` con `(12,)` funciona - la
segunda se estira por filas. `(6,12)` con `(6,)` **falla**, porque alinea 6 contra 12; hay
que volverlo `(6,1)` a mano.

---
## El dominio de trabajo

Los cuatro retos giran alrededor de la misma tabla: las **unidades vendidas de 6 productos
a lo largo de 12 meses** en una tienda de barrio. Filas = productos, columnas = meses.

Es la misma tienda de la Sesion 1 y de la **Actividad 1**, asi que lo que construyas aqui
te sirve despues.

Ejecuta esta celda antes de empezar.

In [ ]:
import numpy as np

PRODUCTOS = np.array(['Arroz', 'Aceite', 'Leche', 'Queso', 'Cafe', 'Panela'])
MESES = np.array(['ene','feb','mar','abr','may','jun',
                  'jul','ago','sep','oct','nov','dic'])

# Unidades vendidas: 6 productos (filas) x 12 meses (columnas)
VENTAS = np.array([
    [120, 135, 128, 142, 150, 133, 141, 156, 149, 138, 160, 172],  # Arroz
    [ 80,  76,  91,  85,  99,  88,  94,  82,  97, 103,  90, 110],  # Aceite
    [210, 198, 225, 240, 232, 219, 245, 251, 238, 260, 249, 271],  # Leche
    [ 45,  52,  38,  61,  49,  55,  43,  58,  47,  63,  50,  66],  # Queso
    [ 95, 102,  88, 115, 121, 108,  99, 130, 142, 118, 125, 137],  # Cafe
    [ 40,  40,  40,  40,  40,  40,  40,  40,  40,  40,  40,  40],  # Panela
])

# Precio de venta por unidad, uno por producto -> forma (6,)
PRECIOS = np.array([3200, 11500, 4800, 12000, 18000, 5400])

# Costo fijo de operacion del mes, uno por mes -> forma (12,)
COSTOS_FIJOS = np.array([820000, 795000, 810000, 850000, 870000, 845000,
                         880000, 905000, 890000, 915000, 930000, 1010000])

print('VENTAS       ', VENTAS.shape, VENTAS.dtype)
print('PRECIOS      ', PRECIOS.shape)
print('COSTOS_FIJOS ', COSTOS_FIJOS.shape)
print('Panela vende exactamente lo mismo todos los meses:', VENTAS[5].std())

---
## Reto 1 - Estandarizar por eje sin reventar

Comparar Leche (vende 200 y pico) contra Queso (vende 50) en crudo no dice nada. Hay que
llevar cada serie a la misma escala.

Escribe `estandarizar(matriz, eje)`.

**Contrato:**
- Devuelve el **z-score** de cada elemento: `(valor - media) / desviacion`, calculando media
  y desviacion **a lo largo del eje indicado**, con `ddof=0` (el de `np.std` por defecto).
- `eje=0` estandariza **por columna** (cada mes contra si mismo).
  `eje=1` estandariza **por fila** (cada producto contra su propia historia).
- Devuelve un array de `float` con **la misma forma** que la entrada.
- **No modifica la matriz de entrada.**
- Si `eje` no es 0 ni 1, lanza `ValueError` y el mensaje debe incluir el valor recibido.

**La parte que importa:** la fila de Panela es constante, asi que su desviacion es `0`.
La formula ingenua te devuelve `nan` o `inf` y ademas te escupe un `RuntimeWarning`.
Contrato: **donde la desviacion sea 0, el resultado es `0.0`**, y el calculo no debe
disparar ninguna division por cero. La verificacion corre tu funcion dentro de un
`np.errstate(divide='raise', invalid='raise')`, asi que si divides por cero se nota.

Sin bucles. `keepdims=True` y `np.where` son tus herramientas.

In [ ]:
def estandarizar(matriz, eje):
    # tu codigo aqui
    pass


In [ ]:
# --- verificacion del reto 1 ---
_copia = VENTAS.copy()
z_fila = estandarizar(VENTAS, 1)
z_col = estandarizar(VENTAS, 0)

assert np.array_equal(VENTAS, _copia), 'La matriz de entrada NO se debe modificar'
assert isinstance(z_fila, np.ndarray), 'Debe devolver un array de NumPy'
assert z_fila.shape == VENTAS.shape, \
    f'Misma forma que la entrada: se esperaba {VENTAS.shape} y llego {z_fila.shape}'
assert z_fila.dtype.kind == 'f', f'Debe ser float, no {z_fila.dtype}'
assert not np.isnan(z_fila).any(), 'Quedaron nan: la fila constante no se esta manejando'
assert not np.isinf(z_fila).any(), 'Quedaron inf: estas dividiendo por una desviacion de 0'

assert np.allclose(z_fila[:5].mean(axis=1), 0), \
    'Con eje=1 cada fila (menos la constante) debe quedar con media 0'
assert np.allclose(z_fila[:5].std(axis=1), 1), \
    'Con eje=1 cada fila (menos la constante) debe quedar con desviacion 1'
assert np.allclose(z_col.mean(axis=0), 0), 'Con eje=0 cada columna debe quedar con media 0'
assert np.allclose(z_col.std(axis=0), 1), 'Con eje=0 cada columna debe quedar con desviacion 1'

assert np.allclose(z_fila[5], 0.0), \
    'Panela es constante: su desviacion es 0, asi que toda la fila debe quedar en 0.0'
assert abs(z_fila[0, 0] - (120 - VENTAS[0].mean()) / VENTAS[0].std()) < 1e-9, \
    'El z-score de Arroz en enero no coincide con (valor - media_fila) / desv_fila'
assert abs(z_col[0, 0] - (120 - VENTAS[:, 0].mean()) / VENTAS[:, 0].std()) < 1e-9, \
    'Con eje=0 la media y la desviacion se toman sobre la COLUMNA, no sobre la fila'

plana = np.full((3, 4), 7.0)
assert np.allclose(estandarizar(plana, 0), 0.0), 'Matriz constante por columna -> todo 0.0'
assert np.allclose(estandarizar(plana, 1), 0.0), 'Matriz constante por fila -> todo 0.0'

try:
    with np.errstate(divide='raise', invalid='raise'):
        estandarizar(plana, 1)
except FloatingPointError:
    raise AssertionError('Estas dividiendo por una desviacion de 0. '
                         'Reemplaza el divisor por 1 antes de dividir, no despues')

try:
    estandarizar(VENTAS, 2)
except ValueError as e:
    assert '2' in str(e), 'El mensaje del ValueError debe incluir el eje recibido'
else:
    raise AssertionError('Un eje distinto de 0 o 1 debe lanzar ValueError')

print('Reto 1 resuelto.')
print('Mejor mes de cada producto en z-score de su propia fila:')
print(np.round(z_fila.max(axis=1), 2))

> **Si te sobra tiempo:** agrega un parametro `metodo` que acepte `'zscore'` o `'minmax'`.
> El min-max lleva cada serie al rango [0,1] y tiene su propio caso degenerado: cuando el
> minimo y el maximo coinciden. Decide que devolver ahi y justificalo.

---
## Reto 2 - Clasificar 72 celdas sin escribir un solo `if`

Escribe `clasificar(ventas, bajo, alto)`.

**Contrato:**
- Devuelve un array de **texto** con la misma forma que `ventas`. Cada celda lleva una de
  estas cuatro etiquetas, evaluadas en este orden de prioridad:

| Etiqueta | Condicion |
|---|---|
| `'flojo'` | la venta es **menor** que `bajo` |
| `'estrella'` | la venta es **mayor o igual** que `alto` **y** ademas supera estrictamente la media de su propia fila |
| `'sostenido'` | la venta es mayor o igual que `alto` pero **no** supera la media de su fila |
| `'normal'` | todo lo demas |

- La media es **por producto**, es decir por fila, calculada sobre la matriz completa que se
  recibe.
- El resultado debe ser un `numpy.ndarray` de tipo texto (`dtype.kind == 'U'`), no una lista
  de listas ni un array de `object`.

**Prohibido bucles y prohibido `if`.** Esto no lo verifica ningun assert: es tu palabra.
Pero es exactamente el punto del reto - `np.where` anidado hace el trabajo de un
`if/elif/elif/else` sobre las 72 celdas a la vez, y una condicion compuesta con `&` sobre
dos arrays de forma distinta te obliga a pensar en broadcasting.

Dos trampas conocidas: `and` no funciona entre arrays (usa `&`, y con parentesis), y los
arrays de texto de NumPy tienen ancho fijo, asi que una etiqueta puede quedar truncada si
armas el resultado a mano.

In [ ]:
def clasificar(ventas, bajo, alto):
    # tu codigo aqui
    pass


In [ ]:
# --- verificacion del reto 2 ---
etq = clasificar(VENTAS, bajo=60, alto=140)

assert isinstance(etq, np.ndarray), 'Debe devolver un array de NumPy, no una lista'
assert etq.shape == VENTAS.shape, \
    f'Misma forma que ventas: se esperaba {VENTAS.shape} y llego {etq.shape}'
assert etq.dtype.kind == 'U', \
    f'Debe ser un array de texto (dtype U), llego {etq.dtype}. Un dtype object no vale'
assert set(np.unique(etq)) <= {'flojo','normal','sostenido','estrella'}, \
    f'Aparecieron etiquetas fuera del contrato: {set(np.unique(etq))}'
assert set(np.unique(etq)) == {'flojo','normal','sostenido','estrella'}, \
    'Con bajo=60 y alto=140 deben aparecer las cuatro etiquetas. Si falta alguna, o la ' \
    'media que usas no es la de cada fila, o el array de texto quedo corto y se trunco'

# Panela vende 40 todos los meses: siempre por debajo de bajo=60
assert (etq[5] == 'flojo').all(), 'Panela vende 40 todos los meses: 40 < 60 es flojo'
# Aceite nunca baja de 60 ni llega a 140
assert (etq[1] == 'normal').all(), \
    'Aceite se mueve entre 76 y 110: ni flojo ni alto, todo normal'

media_arroz = VENTAS[0].mean()   # 143.666...
assert etq[0, 3] == 'sostenido', \
    'Arroz en abril vende 142: pasa el umbral de 140 pero no supera la media de su fila'
assert etq[0, 4] == 'estrella', \
    'Arroz en mayo vende 150: pasa el umbral y supera la media de su fila'
assert etq[0, 0] == 'normal', 'Arroz en enero vende 120: entre 60 y 140'
assert etq[3, 2] == 'flojo', 'Queso en marzo vende 38: por debajo de 60'
assert etq[4, 8] == 'estrella', 'Cafe en septiembre vende 142, y su media de fila es 115'

# la media se toma por FILA, no global
uno = clasificar(np.array([[10, 200], [190, 195]]), bajo=50, alto=100)
assert uno[0, 1] == 'estrella', 'Fila [10,200]: 200 supera su media de 105'
assert uno[1, 0] == 'sostenido', \
    'Fila [190,195]: 190 pasa el umbral pero no supera la media de SU fila (192.5). '\
    'Si te dio estrella, estas usando la media global'
assert uno[0, 0] == 'flojo', 'Fila [10,200]: 10 < 50'

print('Reto 2 resuelto.')
for i, nombre in enumerate(PRODUCTOS):
    print(f'{nombre:8}', ' '.join(f'{e[:4]:>4}' for e in etq[i]))

> **Si te sobra tiempo:** devuelve tambien un diccionario `{etiqueta: conteo}` sin usar
> bucles, con `np.unique(..., return_counts=True)`. Y piensa: si en vez de cuatro etiquetas
> fueran veinte umbrales, `np.where` anidado deja de servir. Busca `np.digitize`.

---
## Reto 3 - Broadcasting de verdad, y el top 3 de cada producto

Dos funciones en este reto. La primera es el broadcasting; la segunda, el indexado avanzado.

### 3a. `margen_mensual(ventas, precios, costos_fijos)`

**Contrato:**
- `ventas` tiene forma `(n_productos, n_meses)`.
- `precios` tiene forma `(n_productos,)`: un precio unitario por **producto**.
- `costos_fijos` tiene forma `(n_meses,)`: el costo de operar en cada **mes**, que se reparte
  **en partes iguales entre los n_productos**.
- Devuelve un array `float` de forma `(n_productos, n_meses)` con
  `ventas[i,j] * precios[i] - costos_fijos[j] / n_productos`.
- Lanza `ValueError` si `precios` no tiene un elemento por producto, o si `costos_fijos` no
  tiene uno por mes. El mensaje debe decir **cual de los dos** no encaja.

Aqui esta el punto del reto: `costos_fijos` con forma `(12,)` se alinea sola contra
`(6,12)`, pero `precios` con forma `(6,)` **no**. NumPy alinea por la derecha y compara 6
contra 12. Te toca decirle que ese 6 es el eje de las filas.

### 3b. `top_meses(matriz, n)`

**Contrato:**
- Devuelve un array de **enteros** de forma `(n_filas, n)` con los **indices de mes** de los
  `n` valores mas altos de cada fila, ordenados de mayor a menor.
- Empate: gana el mes de indice menor.
- Lanza `ValueError` si `n` es mayor que el numero de columnas; el mensaje debe incluir
  ambos numeros.

No devuelvas los valores: devuelve **posiciones**. Y ojo con el empate - por defecto
`np.argsort` usa un algoritmo que no es estable.

Sin bucles en ninguna de las dos.

In [ ]:
def margen_mensual(ventas, precios, costos_fijos):
    # tu codigo aqui
    pass


def top_meses(matriz, n):
    # tu codigo aqui
    pass


In [ ]:
# --- verificacion del reto 3 ---
M = margen_mensual(VENTAS, PRECIOS, COSTOS_FIJOS)
n_prod, n_mes = VENTAS.shape

assert isinstance(M, np.ndarray), 'margen_mensual debe devolver un array'
assert M.shape == VENTAS.shape, \
    f'El margen tiene una fila por producto y una columna por mes: se esperaba ' \
    f'{VENTAS.shape} y llego {M.shape}. Si te dio otra forma, el broadcasting salio mal'
assert M.dtype.kind == 'f', 'El margen es float: hay una division de por medio'

esperado_00 = 120 * 3200 - 820000 / 6
assert abs(M[0, 0] - esperado_00) < 1e-6, \
    f'Arroz en enero: 120 unidades * 3200 - 820000/6 = {esperado_00:.2f}, llego {M[0,0]:.2f}'
esperado_31 = 52 * 12000 - 795000 / 6
assert abs(M[3, 1] - esperado_31) < 1e-6, \
    'Queso en febrero no cuadra: revisa que el precio se aplique por FILA y el costo por COLUMNA'
assert abs(M[4, 11] - (137 * 18000 - 1010000 / 6)) < 1e-6, 'Cafe en diciembre no cuadra'

# si el precio se aplicara por columna en vez de por fila, esta comparacion falla
assert M[4].mean() > M[0].mean(), \
    'Cafe deja mas margen que Arroz porque su precio unitario es 18000 contra 3200. ' \
    'Si no, estas multiplicando el vector de precios sobre el eje equivocado'

try:
    margen_mensual(VENTAS, PRECIOS[:3], COSTOS_FIJOS)
except ValueError as e:
    assert 'precio' in str(e).lower(), \
        'Con precios de largo equivocado, el mensaje debe mencionar precios'
else:
    raise AssertionError('precios con 3 elementos para 6 productos debe lanzar ValueError')

try:
    margen_mensual(VENTAS, PRECIOS, COSTOS_FIJOS[:5])
except ValueError as e:
    assert 'costo' in str(e).lower(), \
        'Con costos_fijos de largo equivocado, el mensaje debe mencionar costos_fijos'
else:
    raise AssertionError('costos_fijos con 5 elementos para 12 meses debe lanzar ValueError')

# --- 3b ---
T = top_meses(VENTAS, 3)
assert isinstance(T, np.ndarray) and T.shape == (n_prod, 3), \
    f'top_meses debe devolver forma ({n_prod}, 3) y llego {getattr(T, "shape", type(T))}'
assert T.dtype.kind in 'iu', f'Son indices: deben ser enteros, no {T.dtype}'

assert list(T[0]) == [11, 10, 7], \
    'Arroz: los tres mejores meses son dic(172), nov(160) y ago(156) -> indices 11, 10, 7. ' \
    'Si te dieron los valores en vez de los indices, te falto argsort'
assert list(T[2]) == [11, 9, 7], 'Leche: dic(271), oct(260), ago(251) -> 11, 9, 7'
assert list(T[5]) == [0, 1, 2], \
    'Panela vende lo mismo todos los meses: con empate total gana el indice menor'

empate = np.array([[5, 9, 9, 2, 9, 1, 9, 9, 9, 9, 9, 9]])
assert list(top_meses(empate, 4)[0]) == [1, 2, 4, 6], \
    'Con valores repetidos deben salir los indices 1, 2, 4, 6 en ese orden. ' \
    'El argsort por defecto no es estable y reordena los empates: pasa kind="stable"'

valores = np.take_along_axis(VENTAS, T, axis=1)
assert (np.diff(valores, axis=1) <= 0).all(), \
    'Los indices deben venir de mayor a menor valor, no de menor a mayor'
assert np.array_equal(valores.max(axis=1), VENTAS.max(axis=1)), \
    'El primer indice de cada fila debe apuntar al maximo de esa fila'

assert list(top_meses(VENTAS, 1)[:, 0]) == list(VENTAS.argmax(axis=1)), \
    'Con n=1 el resultado debe coincidir con argmax por fila'

try:
    top_meses(VENTAS, 20)
except ValueError as e:
    assert '20' in str(e) and '12' in str(e), \
        'El mensaje debe incluir el n pedido (20) y el numero de columnas (12)'
else:
    raise AssertionError('n=20 con 12 meses debe lanzar ValueError')

print('Reto 3 resuelto.')
mejor = top_meses(M, 3)
print('Los tres meses de mayor margen por producto:')
for i, nombre in enumerate(PRODUCTOS):
    print(f'  {nombre:8} {", ".join(MESES[mejor[i]])}')

> **Si te sobra tiempo:** calcula la matriz de distancias entre productos: una matriz
> `(6,6)` donde la celda `(i,j)` es la distancia euclidiana entre los perfiles
> estandarizados de los productos `i` y `j`. Sale en una linea con broadcasting a tres ejes,
> usando `z[:, None, :] - z[None, :, :]`. Piensa que forma tiene ese resultado intermedio
> antes de escribirlo.

---
## Reto 4 - Ventana movil y deteccion de anomalias, sin bucles

El mas duro. Aqui hay que investigar un poco.

Un mes es **anomalo** si se sale del comportamiento reciente de ese mismo producto. Para
medir "comportamiento reciente" se usa una **media movil**: el promedio de los ultimos `k`
meses, incluido el mes actual.

Escribe `detectar_anomalias(ventas, k, z)`.

**Contrato:**
- Devuelve una tupla `(medias, mascara)`.
- `medias` es un array `float` de forma `(n_productos, n_meses)`:
  - `medias[i, t]` es el promedio de `ventas[i, t-k+1 : t+1]`, es decir la ventana de `k`
    meses que **termina** en el mes `t`.
  - Las primeras `k-1` columnas no tienen ventana completa: van en `np.nan`.
- `mascara` es un array de `bool` de la misma forma. Vale `True` cuando
  `abs(ventas[i,t] - medias[i,t]) > z * desviacion_de_la_fila_i`, y `False` en cualquier
  otro caso, **incluidas las primeras `k-1` columnas**.
- La desviacion de la fila se calcula sobre la fila **completa**, con `ddof=0`.
- Si `k` no esta entre 1 y el numero de meses, lanza `ValueError` mencionando `k`.

**Como se hace sin bucles.** Dos caminos, los dos validos:

1. `np.cumsum` sobre el eje de los meses. Si `S` es la suma acumulada con un cero pegado al
   frente, la suma de la ventana que termina en `t` es `S[t+1] - S[t+1-k]`. Todo el vector
   de ventanas sale con una resta de dos rebanadas.
2. `np.lib.stride_tricks.sliding_window_view(x, k, axis=1)`, que te entrega un array con un
   eje extra donde cada posicion es una ventana. Promedias sobre ese eje y listo.

El camino 1 es O(m) y el que se usa en produccion. El 2 es mas legible. Elige, pero entiende
el que elijas: si no sabes por que `S[t+1] - S[t+1-k]` es la suma de la ventana, no lo copies.

Comparar `nan` con `>` devuelve `False` pero deja un `RuntimeWarning`. Evitalo trabajando
solo sobre la parte valida.

In [ ]:
def detectar_anomalias(ventas, k, z):
    # tu codigo aqui
    pass


In [ ]:
# --- verificacion del reto 4 ---
K, Z = 3, 1.0
medias, mascara = detectar_anomalias(VENTAS, K, Z)

assert isinstance(medias, np.ndarray) and isinstance(mascara, np.ndarray), \
    'Debe devolver una tupla de dos arrays: (medias, mascara)'
assert medias.shape == VENTAS.shape, \
    f'medias debe tener la forma de ventas {VENTAS.shape}, y llego {medias.shape}. ' \
    'Rellena con nan las columnas sin ventana completa en vez de recortar el array'
assert mascara.shape == VENTAS.shape, f'mascara debe tener forma {VENTAS.shape}'
assert medias.dtype.kind == 'f', 'medias es float: es un promedio'
assert mascara.dtype == bool, f'mascara debe ser de booleanos, no {mascara.dtype}'

assert np.isnan(medias[:, :K-1]).all(), \
    f'Las primeras {K-1} columnas no tienen ventana completa: deben ser nan'
assert not np.isnan(medias[:, K-1:]).any(), \
    'A partir de la columna k-1 ya hay ventana completa: ahi no puede quedar nan'
assert not mascara[:, :K-1].any(), \
    'Sin ventana completa no se puede declarar anomalia: esas columnas van en False'

assert abs(medias[0, 2] - (120+135+128)/3) < 1e-9, \
    'medias[0,2] debe ser el promedio de ene, feb y mar de Arroz'
assert abs(medias[0, 11] - (138+160+172)/3) < 1e-9, \
    'La ventana TERMINA en el mes t: para dic son oct, nov y dic. ' \
    'Si te dio otro numero, revisa si la centraste o la adelantaste'
assert np.allclose(medias[5, K-1:], 40.0), 'Panela vende 40 fijo: su media movil es 40'

# referencia calculada a mano, con bucles porque aqui si se vale
ref_med = np.full(VENTAS.shape, np.nan)
ref_msk = np.zeros(VENTAS.shape, dtype=bool)
for i in range(VENTAS.shape[0]):
    desv_i = VENTAS[i].std()
    for t in range(K-1, VENTAS.shape[1]):
        m = VENTAS[i, t-K+1:t+1].mean()
        ref_med[i, t] = m
        ref_msk[i, t] = abs(VENTAS[i, t] - m) > Z * desv_i

assert np.allclose(medias[:, K-1:], ref_med[:, K-1:]), \
    'Las medias moviles no coinciden con la referencia calculada mes a mes'
assert np.array_equal(mascara, ref_msk), \
    'La mascara no coincide con la referencia. Revisa que la desviacion sea la de la FILA ' \
    'completa (no la de la ventana) y que la comparacion sea estricta con >'

assert not mascara[5].any(), \
    'Panela es constante: su desviacion es 0 y su diferencia tambien, y 0 > 0 es False'
assert mascara[3, 3], 'Queso en abril salta a 61 con una media movil de 50.3: es anomalo'
assert mascara.sum() == 4, \
    f'Con k=3 y z=1.0 deben salir 4 anomalias en toda la tabla, y salieron {mascara.sum()}'

m1, k1 = detectar_anomalias(VENTAS, 1, 1.0)
assert np.allclose(m1, VENTAS), 'Con k=1 la media movil es el propio valor'
assert not k1.any(), 'Con k=1 la diferencia es 0 en todas partes: no hay anomalias'

try:
    detectar_anomalias(VENTAS, 0, 1.0)
except ValueError as e:
    assert '0' in str(e), 'El mensaje del ValueError debe incluir el k recibido'
else:
    raise AssertionError('k=0 debe lanzar ValueError')

print('Reto 4 resuelto.')
filas, cols = np.nonzero(mascara)
print(f'{len(filas)} meses anomalos con k={K} y z={Z}:')
for i, t in zip(filas, cols):
    print(f'  {PRODUCTOS[i]:8} {MESES[t]}  vendio {VENTAS[i,t]:4}  '
          f'contra una media movil de {medias[i,t]:7.2f}')

> **Si te sobra tiempo:** cambia el criterio a uno robusto - en vez de la media y la
> desviacion, usa la mediana movil y la MAD (mediana de las desviaciones absolutas). La
> mediana movil ya no sale con `cumsum`: ahi `sliding_window_view` deja de ser una
> comodidad y pasa a ser la unica salida decente.

---
## Reto abierto - sin verificacion automatica

Los cuatro retos anteriores tenian una respuesta correcta. Este no.

**El encargo:** la duena de la tienda va a pedir un credito y el banco le pide "un analisis
del ano". Ella te pasa `VENTAS`, `PRECIOS` y `COSTOS_FIJOS`, y nada mas. No sabe que
preguntar.

Escribe `informe_anual(ventas, precios, costos_fijos)` que imprima un informe corto con lo
que **tu** consideres que hay que mirar. Algunas ideas, no todas buenas:

- Que productos sostienen el negocio y cuales solo ocupan espacio
- En que meses la operacion pierde plata
- Tendencia: quien viene creciendo y quien viene cayendo, y como lo mides
- Concentracion del riesgo: cuanto del margen depende de un solo producto
- Estacionalidad comparable entre productos de escalas muy distintas

Se juzga el **criterio**, no la respuesta: que decidiste mirar, por que eso y no otra cosa,
y si el informe le sirve a alguien que no sabe programar. Deja tus decisiones escritas en
comentarios o en la celda de texto que sigue.

Regla que sigue viva: sin bucles para calcular. Los bucles solo para imprimir.

In [ ]:
def informe_anual(ventas, precios, costos_fijos):
    # tu codigo aqui
    pass


informe_anual(VENTAS, PRECIOS, COSTOS_FIJOS)

**Tus decisiones de diseno** (escribe aqui que miraste y por que):

1.
2.
3.

---
## Como se conecta con la Actividad 1

La **Actividad 1 (25%)** se entrega **despues de esta sesion**.
Pide un programa con diccionario de categorias, listas y tuplas de productos, funciones de
total y promedio, un `for` que aplique 10% de descuento a Lacteos, un `if` que clasifique
caro o barato, un **DataFrame de pandas** y un **grafico de matplotlib**.

Lo que hiciste hoy ya cubre el nucleo analitico, y con mejor diseno del que pide la rubrica:

| Punto de la actividad | Donde te quedo |
|---|---|
| Diccionario, listas, tuplas | Sesion 1 |
| Funciones de total y promedio | Sesion 1, reto 1 |
| Iteracion con descuento del 10% | Reto 3: el descuento es un factor sobre el vector de precios |
| Condicional caro / barato | Reto 2: `np.where` es ese `if`, aplicado a toda la tabla |
| DataFrame de pandas | La celda de abajo |
| Grafico de matplotlib | La celda de abajo |
| Codigo comentado | Al armar la entrega |

Un detalle de forma sobre la rubrica: pide explicitamente un `for` y un `if/elif/else`. Si
entregas todo vectorizado, **incluye igual la version con bucle** aunque sea de tres lineas,
o deja escrito por que la reemplazaste. La rubrica la califica alguien que busca esas
palabras.

La celda que sigue es lo minimo de pandas y matplotlib que necesitas. No es un reto: es el
puente. Ejecutala, entiendela y llevatela a tu entrega.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Un DataFrame no es mas que un array 2D con nombres en filas y columnas
df = pd.DataFrame(VENTAS, index=PRODUCTOS, columns=MESES)
df['total_unidades'] = df.sum(axis=1)
df['precio'] = PRECIOS
df['ingreso'] = df['total_unidades'] * df['precio']
df['clase'] = np.where(df['precio'] > 10000, 'caro', 'barato')
print(df[['total_unidades', 'precio', 'ingreso', 'clase']])

# .values te devuelve el array de NumPy que hay debajo: pandas y NumPy son la misma cosa
print('\nTipo de lo que hay debajo del DataFrame:', type(df[MESES].values))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.bar(PRODUCTOS, df['ingreso'] / 1e6)
ax1.set_title('Ingreso anual por producto')
ax1.set_ylabel('millones COP')
ax1.tick_params(axis='x', rotation=45)

ax2.plot(MESES, VENTAS.T)
ax2.set_title('Unidades vendidas por mes')
ax2.set_ylabel('unidades')
ax2.legend(PRODUCTOS, fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

**Lo que falta para poder entregar la Actividad 1**, y que se pierde por forma, no por
fondo:

- Se entrega en **`.ipynb` Y `.py`**, no solo el notebook. En VS Code:
  guarda el notebook como `.ipynb` y crea/guarda tambien el archivo `.py`.
- Sube ambos a una carpeta de **Google Drive con acceso libre de visualizacion**
  (Compartir > Cualquier persona con el enlace > Lector). Si queda restringida, no se puede
  calificar.
- Un documento **Word** con el enlace a esa carpeta.
- En ese Word va la **declaracion de uso de IA**: es obligatoria. Que herramienta usaste,
  para que, y que verificaste tu. Usarla no resta; no declararla si.
- Codigo comentado explicando cada paso.

---
## Para terminar

1. Ejecuta el notebook completo de arriba abajo y confirma que las cuatro verificaciones pasan.
2. Guardalo **con las salidas visibles** (Archivo > Guardar).
3. Si quieres, subelo al aula (opcional, no se califica) y lo reviso y te comento por escrito.

Si terminaste temprano: vuelve al reto 4 y mide con `%timeit` tu version con `cumsum` contra
una con `sliding_window_view` y contra una con dos bucles anidados, sobre una matriz de
1000 x 5000. La diferencia entre las tres es el resumen de toda la sesion.

**Proxima sesion:** pandas en serio - cargar un CSV, limpiar faltantes, detectar outliers.
El reto 4 de hoy fue tu primer detector de outliers: en la Sesion 3 se llama Z-score y viene
en la rubrica de la Actividad 2.

*Dudas: consulta al profesor*